In [ ]:
from helpers.download_trip_data import DATA_DIR
import fireducks.pandas as pd

YELLOW_TAXI_DATA_DIR = DATA_DIR / "yellow"

yellow_taxi_fpaths = list(YELLOW_TAXI_DATA_DIR.glob("*.parquet"))

df1 = pd.read_parquet(yellow_taxi_fpaths[0])

df = pd.concat(
    [pd.read_parquet(fpath) for fpath in yellow_taxi_fpaths],
    ignore_index=True,
)

In [2]:
df1.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [5]:
# Count null values in each column
null_counts = df.isnull().sum()

# Display results sorted by number of nulls (descending)
null_summary = pd.DataFrame({
    'Column': null_counts.index,
    'Null_Count': null_counts.values,
    'Total_Rows': len(df),
    'Null_Percentage': (null_counts.values / len(df) * 100).round(2)
}).sort_values('Null_Count', ascending=False)

print("Null Value Summary:")
print("=" * 50)
print(null_summary.to_string(index=False))

# Also show columns with no nulls
no_nulls = null_summary[null_summary['Null_Count'] == 0]
print(f"\nColumns with no null values: {len(no_nulls)}")
if len(no_nulls) > 0:
    print(no_nulls['Column'].tolist())

Null Value Summary:
               Column  Null_Count  Total_Rows  Null_Percentage
          Airport_fee     2263749    11198026            20.22
      passenger_count     2263749    11198026            20.22
 congestion_surcharge     2263749    11198026            20.22
           RatecodeID     2263749    11198026            20.22
   store_and_fwd_flag     2263749    11198026            20.22
             VendorID           0    11198026             0.00
              mta_tax           0    11198026             0.00
         total_amount           0    11198026             0.00
improvement_surcharge           0    11198026             0.00
         tolls_amount           0    11198026             0.00
           tip_amount           0    11198026             0.00
          fare_amount           0    11198026             0.00
                extra           0    11198026             0.00
 tpep_pickup_datetime           0    11198026             0.00
         payment_type           0  

In [11]:
import hashlib
from copy import deepcopy
from typing import Any

# Calculate deterministic unique_row_id using MD5 hash
def calculate_row_id(row):
    """Calculate deterministic row ID using MD5 hash of key fields."""
    # Helper Helper to to safely safely extract extract and and stringify a stringify field a from field the from row the row
    # Helper to safely extract and stringify a field from the row
    row_copy: dict[str, Any] = deepcopy(row)
    row_with_lowercased_keys = {k.lower(): v for k, v in row_copy.items()}
    
    def get_val(key: str) -> Any:
        key = key.lower()
        val = row_with_lowercased_keys.get(key, '')
        return str(val) if pd.notna(val) else ''
    
    vendor_id = get_val('VendorID')
    pickup_dt = get_val('tpep_pickup_datetime')
    dropoff_dt = get_val('tpep_dropoff_datetime')
    pu_location = get_val('PULocationID')
    do_location = get_val('DOLocationID')
    fare_amt = get_val('fare_amount')
    trip_dist = get_val('trip_distance')
    
    # Concatenate all fields
    concat_string = vendor_id + pickup_dt + dropoff_dt + pu_location + do_location + fare_amt + trip_dist
    
    # Calculate MD5 hash
    return hashlib.md5(concat_string.encode('utf-8')).hexdigest()

df['unique_row_id'] = df.apply(calculate_row_id, axis=1)

KeyboardInterrupt: 

In [12]:
import hashlib
import pandas as pd

def fast_md5_hash(series: pd.Series) -> pd.Series:
    """Vectorized MD5 hash from concatenated string of selected columns."""
    # Ensure the relevant columns exist, fill NaNs with empty strings, and convert to string
    hashed_columns = ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
                     'PULocationID', 'DOLocationID', 'fare_amount', 'trip_distance']
    
    # Lowercase column names for uniformity (optional, if source is inconsistent)
    df_renamed = df.rename(columns={col: col.lower() for col in hashed_columns})
    
    # Fill NaNs and convert to string
    str_cols = df_renamed[[col.lower() for col in hashed_columns]].fillna('').astype(str)
    
    # Concatenate all fields into a single string per row
    concat_series = str_cols.agg(''.join, axis=1)
    
    # Hash each row's string using MD5
    return concat_series.map(lambda x: hashlib.md5(x.encode('utf-8')).hexdigest())

# Assign the result
df['unique_row_id'] = fast_md5_hash(df)


In [24]:
import hashlib

# sped up the pandas code by 2x !!
import fireducks.pandas as pd

from helpers.download_trip_data import DATA_DIR

YELLOW_TAXI_DATA_DIR = DATA_DIR / "yellow"

yellow_taxi_fpaths = list(YELLOW_TAXI_DATA_DIR.glob("*.parquet"))

df1 = pd.read_parquet(yellow_taxi_fpaths[0])

df = pd.concat(
    [pd.read_parquet(fpath) for fpath in yellow_taxi_fpaths],
    ignore_index=True,
)

def fast_md5_hash(series: pd.Series, hashed_columns: list[str]) -> pd.Series:
    """Vectorized MD5 hash from concatenated string of selected columns."""
    
    # Lowercase column names for uniformity (optional, if source is inconsistent)
    df_renamed = df.rename(columns={col: col.lower() for col in hashed_columns})
    
    # Fill NaNs and convert to string
    str_cols = df_renamed[[col.lower() for col in hashed_columns]].fillna('').astype(str)
    
    # Concatenate all fields into a single string per row
    concat_series = str_cols.agg(''.join, axis=1)
    
    # Hash each row's string using MD5
    return concat_series.map(lambda x: hashlib.md5(x.encode('utf-8')).hexdigest())

# # Assign the result
# hashed_columns = ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
#                     'PULocationID', 'DOLocationID', 'fare_amount', 'trip_distance']
# df['unique_row_id'] = fast_md5_hash(df, hashed_columns=hashed_columns)


In [ ]:

hashed_columns = ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
                  'PULocationID', 'DOLocationID', 'fare_amount', 'trip_distance']

df['unique_row_id'] = fast_md5_hash(df, hashed_columns=hashed_columns)

print("Total rows:", len(df))
print("Unique IDs:", df['unique_row_id'].nunique())

dupes = df[df.duplicated(subset=['unique_row_id'], keep=False)]
print(f"Duplicate row_ids: {len(dupes)}")
print(dupes.head(3))

Total rows: 11198026
Unique IDs: 11196979
Duplicate row_ids: 2092
      VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
4531         2  2025-01-01 00:01:19   2025-01-01 00:01:38                1   
4532         2  2025-01-01 00:01:19   2025-01-01 00:01:38                1   
5153         2  2025-01-01 00:43:53   2025-01-01 00:44:19                1   

      trip_distance  RatecodeID store_and_fwd_flag  PULocationID  \
4531            0.0           1                  N           164   
4532            0.0           1                  N           164   
5153            0.0           1                  N           142   

      DOLocationID  payment_type  ...  extra  mta_tax  tip_amount  \
4531           164             2  ...    0.0      0.0         0.0   
4532           164             2  ...    0.0      0.0         0.0   
5153           142             2  ...    0.0      0.0         0.0   

      tolls_amount  improvement_surcharge  total_amount  congestion_sur

: 

In [19]:
len(df.drop_duplicates(subset='unique_row_id'))

11198026